In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

### SET BASE_DIR ###
BASE_DIR = '/data'

# Define the root directory where the models and outputs are saved
save_root = os.path.join(BASE_DIR, "CEBRA_project/Scripts_to_publish/xCEBRA_models_to_publish")

# Choosing the correct jacobian regularizer weight and one run
target_weight = "10p0"
target_run = "run_0"

# Only plot the first three timepoints 
timepoints = [0, 1, 2]  

# Function to load and plot the data for a specific weight and run
def plot_attribution_maps_for_weight_and_run(save_root, target_weight, target_run, timepoints, jacobian, save_suffix):
    # Construct the path to the specific weight directory
    weight_dir = os.path.join(save_root, f"regularizer_weight_{target_weight}")
    save_dir = os.path.join(BASE_DIR, "CEBRA_project/Scripts_to_publish/Plotting_for_figures/Figure_1")

    if not os.path.exists(weight_dir):
        print(f"Weight directory {weight_dir} does not exist. Exiting.")
        return
    
    # Construct the path to the specific run directory
    run_dir = os.path.join(weight_dir, target_run)
    
    if not os.path.exists(run_dir):
        print(f"Run directory {run_dir} does not exist. Exiting.")
        return
    
    # Load the jacobian mean file for this run
    jacobian_file = os.path.join(run_dir, jacobian)
    
    if os.path.exists(jacobian_file):
        # Load the data from the file
        jacobian_data = np.load(jacobian_file)
        
        # Print the keys to check what arrays are available
        print(f"Keys in {jacobian_file}: {list(jacobian_data.keys())}")
        
        # Access the array stored in the file 
        jf_array = jacobian_data.get('arr_0')  
        
        if jf_array is not None:
            # Extract the data for the selected timepoints
            for idx, timepoint in enumerate(timepoints):
                if timepoint < jf_array.shape[-1]:
                    data_at_timepoint = jf_array[timepoint, :, :]
                    
                    # Set figure size
                    plt.figure(figsize=(20, 5)) 

                    # Plot the heatmap without axes, colorbar, etc. with the given transparency
                    plt.matshow(np.abs(data_at_timepoint),cmap="Blues", aspect="auto", fignum=False)
                    plt.axis('off')  # Turn off axes
                    plt.gca().patch.set_facecolor('none')  # No background color (remove white box)
                    plt.tight_layout()

                    # Save the heatmap with transparency and suffix
                    plt.savefig(os.path.join(save_dir, f"{save_suffix}_map_timepoint_{timepoint}_heatmap.svg"), bbox_inches='tight', transparent=True)
                    plt.close()
                    print(f"Saved heatmap for {jacobian_file} at timepoint {timepoint}")
                else:
                    print(f"Timepoint {timepoint} is out of range for {jacobian_file}")
        else:
            print(f"Array 'arr_0' not found in {jacobian_file}. Skipping.")
    else:
        print(f"File {jacobian_file} does not exist in {save_dir}. Skipping.")

    # Plot the colorbar separately using the same colormap as the heatmap
    for idx, timepoint in enumerate(timepoints):
        # Create a figure with the same width-to-height ratio
        plt.figure(figsize=(12.5, 5))
        
        # Create an invisible image to base the colorbar on
        im = plt.imshow(np.random.rand(10, 10), cmap="Blues")  # Dummy image with same colormap
        plt.colorbar(im)  # Add colorbar based on this dummy image
        
        # Remove the ticks and labels for the colorbar
        colorbar = plt.colorbar(im)
        colorbar.set_ticks([])  # Remove ticks
        colorbar.outline.set_visible(False)  # Remove the outline (frame) of the colorbar
        
        # Turn off axes for colorbar
        plt.axis('off')
        
        # Save the colorbar with transparency and suffix
        plt.savefig(os.path.join(save_dir, f"{save_suffix}_colorbar_timepoint_{timepoint}.svg"), bbox_inches='tight', transparent=True)
        plt.close()
        print(f"Saved colorbar for timepoint {timepoint}")




# Run the function to plot the attribution maps for the specified weight, run, and timepoints
plot_attribution_maps_for_weight_and_run(save_root, target_weight, target_run, timepoints, jacobian="jacobian_raw_jf-convabs-inv-svd.npz", save_suffix = "jf-convabs-inv-svd")






In [ ]:
plot_attribution_maps_for_weight_and_run(save_root, target_weight, target_run, timepoints,jacobian = "jacobian_raw_jf.npz",save_suffix = "jf")

# Plot pipeline for Figure S2

In [ ]:
# Apply dark theme
plt.style.use('dark_background')

# ==========================================
# Configuration 
# ==========================================
WEIGHT = 10.0
WTAG = "10p0"  
SUFFIX = "binary_mean"
ROOT = os.path.join(BASE_DIR, "CEBRA_project/Scripts_to_publish/xCEBRA_models_to_publish")
SAVE_DIR = os.path.join(BASE_DIR, "CEBRA_project/Scripts_to_publish/Plotting_for_figures/Figure_S2")

# Ensure the output directory exists
os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================================
# Safe Loader
# ==========================================
def load_map(path):
    """Safely loads .npy or .npz files without modifying them."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    
    if path.endswith(".npz"):
        z = np.load(path)
        if "arr_0" in z.files:
            return z["arr_0"]
        return z[z.files[0]]
    return np.load(path)

# ==========================================
# Helper: Isolated Plotting & Saving
# ==========================================
def plot_isolated_heatmap(data, title, cmap, save_path, vmin=None, vmax=None):
    """Creates a single, standalone figure and saves it to disk."""
    # Enforce 4:1 width-to-height ratio
    fig, ax = plt.subplots(figsize=(8, 2), dpi=150)
    if data.shape[0] > data.shape[1]: 
        data = data.T 
        
    im = ax.imshow(data, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=12, color='white', pad=10)
    
    # Strip axes ticks and descriptions
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Maintain a clean border for dark mode
    for spine in ax.spines.values():
        spine.set_color('gray')
        
    cbar = plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04)
    cbar.ax.tick_params(colors='gray')
    cbar.outline.set_edgecolor('gray')
    
    plt.tight_layout()
    
    # Save the figure as a transparent SVG
    plt.savefig(save_path, bbox_inches='tight', transparent=True)
    plt.close(fig)  # Close the figure to free memory
    print(f"Saved: {os.path.basename(save_path)}")

# ==========================================
# 1. The Alignment Pipeline (3 Runs, Isolated)
# ==========================================
def plot_pipeline_isolated(runs=(0, 1, 2)):
    """Plots every step of the pipeline for multiple runs and saves them."""
    
    pipeline_steps = [
        ("1_TimeAveraged", "jacobian_mean_jf-convabs-inv-svd.npz", "Blues"),
        ("2_ProcrustesAligned", "jacobian_mean_jf-convabs-inv-svd_cebra_procrustes.npy", "Blues"),
        ("3_Binarized_Z0", "jacobian_mean_jf-convabs-inv-svd_cebra_procrustes_binary_z0.npy", "Blues")
    ]
    
    for row_label, filename, cmap in pipeline_steps:
        for run_idx in runs:
            run_dir = os.path.join(ROOT, f"regularizer_weight_{WTAG}", f"run_{run_idx}")
            filepath = os.path.join(run_dir, filename)
            
            # Format a clean title
            step_name = row_label.split("_")[1]
            if "Aligned" in step_name and run_idx == runs[0]:
                title = f"{step_name} (Run {run_idx} Ref)"
            else:
                title = f"{step_name} (Run {run_idx})"
                
            try:
                data = load_map(filepath)
                # Create a safe filename without spaces or parentheses
                safe_name = f"Pipeline_{step_name}_Run{run_idx}.svg"
                if "Ref" in title: safe_name = safe_name.replace(".svg", "_Ref.svg")
                save_path = os.path.join(SAVE_DIR, safe_name)
                
                plot_isolated_heatmap(data, title, cmap, save_path)
            except FileNotFoundError:
                print(f"[MISSING] {title}: {filepath}")

    # Plot Final Consensus Map standalone
    consensus_path = os.path.join(ROOT, f"jf_convabs_inv_svd_cebra_procrustes_{SUFFIX}.npy")
    try:
        data = load_map(consensus_path)
        title = "4_Final Consensus Map (Averaged)"
        save_path = os.path.join(SAVE_DIR, "Pipeline_4_Final_Consensus_Map.svg")
        plot_isolated_heatmap(data, title, "Blues", save_path, vmin=0, vmax=1)
        
        # Align data shape to (Dimensions, Features) before collapsing
        data_aligned = data.T if data.shape[0] > data.shape[1] else data
        
        # Collapse over dimensions (axis=0) to create a 1D probability map
        collapsed_data = data_aligned.mean(axis=0, keepdims=True) 
        
        title_collapsed = "5_Final Consensus Map (Collapsed over Dimensions)"
        save_path_collapsed = os.path.join(SAVE_DIR, "Pipeline_5_Final_Consensus_Map_Collapsed.svg")
        plot_isolated_heatmap(collapsed_data, title_collapsed, "Blues", save_path_collapsed, vmin=0, vmax=1)
        # =======================================================
        
    except FileNotFoundError:
        print(f"[MISSING] Consensus Map: {consensus_path}")

# ==========================================
# 2. The Raw Extraction Methods 
# ==========================================
def plot_extraction_methods_isolated(run_idx=0):
    """Plots the 3 different raw Jacobian maps and saves them."""
    run_dir = os.path.join(ROOT, f"regularizer_weight_{WTAG}", f"run_{run_idx}")
    
    methods = {
        "JF_Raw": "jacobian_mean_jf.npz",
        "JF_InvSVD": "jacobian_mean_jf-inv-svd.npz",
        "JF_ConvAbsInvSVD": "jacobian_mean_jf-convabs-inv-svd.npz"
    }
    
    for title_key, filename in methods.items():
        filepath = os.path.join(run_dir, filename)
        title = f"{title_key} (Run {run_idx})"
        
        try:
            data = load_map(filepath)
            safe_name = f"Extraction_{title_key}_Run{run_idx}.svg"
            save_path = os.path.join(SAVE_DIR, safe_name)
            
            plot_isolated_heatmap(data, title, "Blues", save_path)
        except FileNotFoundError:
            print(f"[MISSING] {title}: {filepath}")

# ==========================================
# Execution
# ==========================================
if __name__ == "__main__":
    print(f"Saving interactive plots (4:1 Aspect Ratio) to:\n{SAVE_DIR}\n")
    
    # 1. Generate individual pipeline plots
    plot_pipeline_isolated(runs=[0, 1, 2])
    
    # 2. Generate individual extraction method plots
    plot_extraction_methods_isolated(run_idx=0)
    
    print("\n✓ Finished saving all plots.")